<a href="https://colab.research.google.com/github/sunayan1/QuantRag/blob/main/building_triplets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q pypdf langchain langchain-text-splitters sentence-transformers faiss-cpu tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.8 MB/s eta 0:00:00


In [ ]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (549 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [ ]:
!which ollama


/usr/local/bin/ollama


In [ ]:
import subprocess, time
process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)
!ollama pull llama3.1:8b

In [ ]:
!pip uninstall -y nltk
!pip install -q nltk

Found existing installation: nltk 3.9.1
Uninstalling nltk-3.9.1:
  Successfully uninstalled nltk-3.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.1 MB/s eta 0:00:00


In [ ]:
from posixpath import basename
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import glob, os
import requests, json

pdf_dir = "/content/drive/MyDrive/disaster_pdf"
pdf_paths = glob.glob(os.path.join(pdf_dir, "*.pdf"))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

all_chunks = []

prompt = """Look at these text and distinguish if the text is SUBSTANTIVE CONTENT(real information text such as policy explanation, definitions, data, measures and all the information related to disaster management)
or Non useful content( such as ministry offices name or any other office name, table of content, page header, page bottoms, or any other information that is irrelevant to disaster management)
Respond with ONLY one word: CONTENT or NONCONTENT"""

def filter_layer(chunk, model="llama3.1:8b"):
    full_prompt = f"{prompt} \n Text:{chunk}"
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": full_prompt,
        "stream": False,
        "options": {"temperature": 0}
    })
    result = resp.json()["response"].strip().upper()
    if result.startswith("CONTENT") and not result.startswith("NONCONTENT"):
        return chunk
    return None

for j, path in enumerate(pdf_paths):
    reader = PdfReader(path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    chunks = splitter.split_text(full_text)

    # progress bar over chunks for THIS pdf
    for i, c in enumerate(tqdm(chunks, desc=f"[{j+1}/{len(pdf_paths)}] {os.path.basename(path)}")):
        filtered_chunk = filter_layer(c)
        if filtered_chunk is None:
            continue

        filtered_chunk = filtered_chunk.strip()
        if len(filtered_chunk) < 50:
            continue

        all_chunks.append({
            "text": filtered_chunk,
            "source": os.path.basename(path),
            "chunk_id": f"{os.path.basename(path)}_{i}"
        })

    tqdm.write(f"Pdf position: {j + 1} Total chunks so far: {len(all_chunks)}")

[1/1] 1476.pdf: 100%|██████████| 75/75 [02:23<00:00,  1.92s/it]

Pdf position: 1 Total chunks so far: 68


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in all_chunks]
embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f"first embeddings: {embeddings}")
embeddings = np.array(embeddings).astype("float32")
print(f"Second embeddings: {embeddings}")

index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product on normalized vecs = cosine sim
index.add(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

first embeddings: [[ 0.04231879  0.04281643  0.04322741 ... -0.05758212 -0.04549469
   0.02629396]
 [ 0.02771887  0.02858086  0.13753818 ... -0.06610915  0.01437662
   0.00279185]
 [ 0.04486959  0.037604    0.11531496 ... -0.06748806 -0.0579324
   0.0327023 ]
 ...
 [ 0.05860799  0.02210936  0.03727939 ... -0.00212064 -0.0547094
  -0.03272353]
 [ 0.0478498   0.04560095  0.01327428 ... -0.01889988 -0.05796284
  -0.01122713]
 [ 0.00847169  0.07959674  0.02648012 ... -0.06044812 -0.06742106
   0.02056222]]
Second embeddings: [[ 0.04231879  0.04281643  0.04322741 ... -0.05758212 -0.04549469
   0.02629396]
 [ 0.02771887  0.02858086  0.13753818 ... -0.06610915  0.01437662
   0.00279185]
 [ 0.04486959  0.037604    0.11531496 ... -0.06748806 -0.0579324
   0.0327023 ]
 ...
 [ 0.05860799  0.02210936  0.03727939 ... -0.00212064 -0.0547094
  -0.03272353]
 [ 0.0478498   0.04560095  0.01327428 ... -0.01889988 -0.05796284
  -0.01122713]
 [ 0.00847169  0.07959674  0.02648012 ... -0.06044812 -0.06742106

In [ ]:
!pkill -9 ollama
time.sleep(2)

import subprocess, time
process = subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(8)
!curl http://localhost:11434/api/tags

{"models":[{"name":"llama3.1:8b","model":"llama3.1:8b","modified_at":"2026-07-30T02:42:46.200989956Z","size":4920753328,"digest":"46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":4096},"capabilities":["completion","tools"]}]}

In [ ]:
!dmesg -T | grep -i -E "killed process|out of memory" | tail -20

In [ ]:
import requests, json
from tqdm import tqdm

def ensure_ollama_running():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        return
    except requests.exceptions.RequestException:
        pass
    tqdm.write("ollama not responding — restarting it")
    subprocess.Popen(["/usr/local/bin/ollama", "serve"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=3)
            return
        except requests.exceptions.RequestException:
            time.sleep(2)
    raise RuntimeError("ollama would not come back up")

def generate_query(chunk_text, model="llama3.1:8b"):
    prompt = f"""
You are simulating a person who vaguely remembers something about disaster management but doesn't recall exact details or terminology.

Read the passage below and write ONE short, vague, natural-sounding question a person might ask, as if searching for this information — do NOT quote exact phrases from the passage, and do NOT make it too specific or textbook-like.

Passage:
\"\"\"{chunk_text}\"\"\"

Respond with ONLY the question, nothing else."""

    try:
        resp = requests.post("http://localhost:11434/api/generate", json={
            "model": model, "prompt": prompt, "stream": False,
            "options": {"temperature": 0.8, "num_predict": 60, "repeat_penalty": 1.3}
        }, timeout=45)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except requests.exceptions.RequestException as e:
        tqdm.write(f"generate_query failed: {e}")
        return None


for c in tqdm(all_chunks):
  ensure_ollama_running()
  c["query"] = generate_query(c["text"])

100%|██████████| 68/68 [01:40<00:00,  1.47s/it]


In [ ]:
import random
for c in random.sample(all_chunks, 5):
    print("CHUNK:", c["text"][:150], "...")
    print("QUERY:", c["query"])
    print("---")

CHUNK: reduction and management activities.
7 .32  Disaster resilient community will be 
developed by diversifying means of 
livelihoods.
7 .33 Flood, inunda ...
QUERY: Is it true that they're trying to create farming systems in our community that can handle extreme weather conditions?
---
CHUNK: 5.6.  To ensure “Build Back Better” approach 
for post-disaster recovery, rehabilitation 
and reconstruction.
6. concept
Disaster risk reduction natio ...
QUERY: What's supposed to happen in this "Build Back Better" approach after a disaster?
---
CHUNK: international community during big 
disaster.
7 .52  Trauma care centers will be established 
in major cities.
23
National Policy for Disaster Risk Re ...
QUERY: Do they have some kind of plan in place to help people recover from disasters?
---
CHUNK: infrastructure and other physical 
infrastructure including water supply.
7 .40  Programs on disaster risk reduction 
and management will be conducted ...
QUERY: How does a country make sure it's 

In [ ]:
def get_negative(idx, k=5):
    query_vec = embeddings[idx].reshape(1, -1)
    scores, neighbors = index.search(query_vec, k + 1)
    for n in neighbors[0]:
        if n != idx:
            return all_chunks[n]
    return None

for i, c in enumerate(tqdm(all_chunks)):
    neg = get_negative(i)
    c["negative"] = neg["text"]
    c["negative_source"] = neg["source"]

100%|██████████| 68/68 [00:00<00:00, 9633.29it/s]


In [ ]:
import pandas as pd

dataset = [{
    "query": c["query"],
    "positive": c["text"],
    "negative": c["negative"],
    "positive_source": c["source"],
    "negative_source": c["negative_source"]
} for c in all_chunks]

df = pd.DataFrame(dataset)
df.to_json("/content/drive/MyDrive/disaster_vqc_dataset.jsonl", orient="records", lines=True)
df.to_csv("/content/drive/MyDrive/disaster_vqc_dataset.csv", index=False)

df.head()

,query,positive,negative,positive_source,negative_source
0,What kind of threats does Nepal typically face...,10\nNational Policy for Disaster Risk Reductio...,“Nepal Disaster Report” .\n28\nNational Policy...,1476.pdf,1476.pdf
1,Do we have plans in place to deal with some of...,from great loss of human lives and damage \nto...,"hailstorm, avalanche, glacial lake outburst, \...",1476.pdf,1476.pdf
2,What kinds of events are more likely to happen...,"hailstorm, avalanche, glacial lake outburst, \...",from great loss of human lives and damage \nto...,1476.pdf,1476.pdf
3,What kinds of things are considered hazards th...,"and micro-organism havoc, animal and bird \nin...",10\nNational Policy for Disaster Risk Reductio...,1476.pdf,1476.pdf
4,Do you think there's a way to prepare for natu...,"poisonous food consumption, environmental \npo...",from great loss of human lives and damage \nto...,1476.pdf,1476.pdf
